In [1]:
# =============== HYBRID INFERENCE CODE (Run with Internet OFF) ===============
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm
import numpy as np
import pandas as pd
from PIL import Image
import os
from tqdm import tqdm

class Config:
    TEST_CSV = '/kaggle/input/csiro-biomass/test.csv'
    TEST_IMG_DIR = '/kaggle/input/csiro-biomass/test'
    MODEL_DIR = '/kaggle/input/vit-hybrid-model1/'  # ← REPLACE WITH YOUR DATASET NAME IF NEEDED
    CNN_BACKBONE = 'convnextv2_base.fcmae_ft_in22k_in1k'
    TRANSFORMER_BACKBONE = 'swinv2_base_window16_256'  
    IMG_SIZE = 256
    TARGETS = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    FUSION_METHOD = 'attention'

class AttentionFusion(nn.Module):
    def __init__(self, cnn_dim, transformer_dim, hidden_dim=768): 
        super().__init__()
        self.cnn_proj = nn.Linear(cnn_dim, hidden_dim)
        self.transformer_proj = nn.Linear(transformer_dim, hidden_dim)
        self.attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, dropout=0.1, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim * 2, hidden_dim)
        )
    
    def forward(self, cnn_features, transformer_features):
        cnn_proj = self.cnn_proj(cnn_features).unsqueeze(1)
        trans_proj = self.transformer_proj(transformer_features).unsqueeze(1)
        features = torch.cat([cnn_proj, trans_proj], dim=1)
        attended, _ = self.attention(features, features, features)
        features = self.norm(features + attended)
        output = features + self.ffn(features)
        return output.mean(dim=1)

class HybridConvViTBiomass(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn_backbone = timm.create_model(Config.CNN_BACKBONE, pretrained=False, num_classes=0, global_pool='avg')
        self.transformer_backbone = timm.create_model(Config.TRANSFORMER_BACKBONE, pretrained=False, num_classes=0, global_pool='avg')
        
        if Config.FUSION_METHOD == 'attention':
            self.fusion = AttentionFusion(self.cnn_backbone.num_features, self.transformer_backbone.num_features, hidden_dim=768)
            fused_features = 768
        elif Config.FUSION_METHOD == 'concat':
            self.fusion = None
            fused_features = self.cnn_backbone.num_features + self.transformer_backbone.num_features
        else:
            assert self.cnn_backbone.num_features == self.transformer_backbone.num_features
            self.fusion = None
            fused_features = self.cnn_backbone.num_features
        
        self.head = nn.Sequential(
            nn.Linear(fused_features, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.175),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.125),
            nn.Linear(256, 5)
        )
    
    def forward(self, x):
        cnn_feat = self.cnn_backbone(x)
        transformer_feat = self.transformer_backbone(x)
        if Config.FUSION_METHOD == 'attention':
            fused = self.fusion(cnn_feat, transformer_feat)
        elif Config.FUSION_METHOD == 'concat':
            fused = torch.cat([cnn_feat, transformer_feat], dim=1)
        else:
            fused = cnn_feat + transformer_feat
        return self.head(fused)

class TestDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        img = Image.open(os.path.join(self.img_dir, img_path)).convert('RGB')
        return self.transform(img), img_path

eval_transform = transforms.Compose([
    transforms.Resize(Config.IMG_SIZE),
    transforms.CenterCrop(Config.IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load test data
df_test = pd.read_csv(Config.TEST_CSV)
df_test['image_path'] = df_test['image_path'].str.replace(r'^test/', '', regex=True)
df_unique = df_test[['image_path']].drop_duplicates().reset_index(drop=True)

# Load models and collect normalization stats
model_paths = [f"{Config.MODEL_DIR}hybrid_convvit_fold_{i}_final.pth" for i in range(1, 4)]  # Only 3 folds
models_list = []
all_means = []
all_stds = []

for path in model_paths:
    print(f"Loading {path}...")
    model = HybridConvViTBiomass().to(Config.DEVICE)
    checkpoint = torch.load(path, map_location=Config.DEVICE, weights_only=False)
    state_dict = checkpoint['model_state_dict']
    
    # Fix 1: Remove 'module.' prefix (if trained with DataParallel)
    cleaned_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith('module.'):
            cleaned_state_dict[k[7:]] = v
        else:
            cleaned_state_dict[k] = v
    
    # Fix 2: Remap ConvNeXtV2 stem keys for timm version compatibility
    # Older timm versions used 'stem.conv'/'stem.norm' instead of 'stem.0'/'stem.1'
    remapped_state_dict = {}
    for k, v in cleaned_state_dict.items():
        if 'cnn_backbone.stem.0' in k:
            remapped_state_dict[k.replace('stem.0', 'stem.conv')] = v
        elif 'cnn_backbone.stem.1' in k:
            remapped_state_dict[k.replace('stem.1', 'stem.norm')] = v
        else:
            remapped_state_dict[k] = v
    
    # Fix 3: Load with strict=False to tolerate minor mismatches
    missing_keys, unexpected_keys = model.load_state_dict(remapped_state_dict, strict=False)
    
    # Optional: Debug output (comment out for silent submission)
    if missing_keys:
        print(f" Missing keys (first 3): {missing_keys[:3]}")
    if unexpected_keys:
        print(f" Unexpected keys (first 3): {unexpected_keys[:3]}")
    
    model.eval()
    models_list.append(model)
    all_means.append(checkpoint['target_mean'])
    all_stds.append(checkpoint['target_std'])

# Compute ensemble normalization stats
ensemble_mean = np.mean(all_means, axis=0)
ensemble_std = np.mean(all_stds, axis=0)

# Inference
test_ds = TestDataset(df_unique, Config.TEST_IMG_DIR, eval_transform)
test_dl = DataLoader(test_ds, batch_size=4, shuffle=False)

all_img_paths, all_preds = [], []
with torch.no_grad():
    for imgs, paths in tqdm(test_dl, desc="Inference"):
        # Get predictions from all models
        model_preds = []
        for model in models_list:
            pred = model(imgs.to(Config.DEVICE))
            model_preds.append(pred)
        
        # Average predictions across models
        ensemble_pred = torch.stack(model_preds).mean(dim=0)
        all_preds.append(ensemble_pred.cpu())
        all_img_paths.extend(paths)

# Concatenate all predictions
preds_normalized = torch.cat(all_preds, dim=0).numpy()

# Denormalize using ensemble stats
preds_denormalized = preds_normalized * ensemble_std + ensemble_mean

# Build submission
rows = []
for i, img_path in enumerate(all_img_paths):
    img_id = os.path.splitext(os.path.basename(img_path))[0]
    for j, target in enumerate(Config.TARGETS):
        sample_id = f"{img_id}__{target}"
        # Ensure non-negative predictions
        prediction = float(max(0.0, preds_denormalized[i, j]))
        rows.append({"sample_id": sample_id, "target": prediction})

# Save submission
submission = pd.DataFrame(rows)[['sample_id', 'target']]
submission.to_csv('submission.csv', index=False)
print("\n submission.csv saved!")
print(f"Submission shape: {submission.shape}")
print(submission.head())

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Loading /kaggle/input/vit-hybrid-model1/hybrid_convvit_fold_1_final.pth...
 Missing keys (first 3): ['cnn_backbone.stem.0.weight', 'cnn_backbone.stem.0.bias', 'cnn_backbone.stem.1.weight']
 Unexpected keys (first 3): ['cnn_backbone.stem.conv.weight', 'cnn_backbone.stem.conv.bias', 'cnn_backbone.stem.norm.weight']
Loading /kaggle/input/vit-hybrid-model1/hybrid_convvit_fold_2_final.pth...
 Missing keys (first 3): ['cnn_backbone.stem.0.weight', 'cnn_backbone.stem.0.bias', 'cnn_backbone.stem.1.weight']
 Unexpected keys (first 3): ['cnn_backbone.stem.conv.weight', 'cnn_backbone.stem.conv.bias', 'cnn_backbone.stem.norm.weight']
Loading /kaggle/input/vit-hybrid-model1/hybrid_convvit_fold_3_final.pth...
 Missing keys (first 3): ['cnn_backbone.stem.0.weight', 'cnn_backbone.stem.0.bias', 'cnn_backbone.stem.1.weight']
 Unexpected keys (first 3): ['cnn_backbone.stem.conv.weight', 'cnn_backbone.stem.conv.bias', 'cnn_backbone.stem.norm.weight']


Inference: 100%|██████████| 1/1 [00:03<00:00,  3.21s/it]


 submission.csv saved!
Submission shape: (5, 2)
                    sample_id     target
0  ID1001187975__Dry_Clover_g   0.000000
1    ID1001187975__Dry_Dead_g  26.015911
2   ID1001187975__Dry_Green_g  35.020622
3   ID1001187975__Dry_Total_g  59.236315
4         ID1001187975__GDM_g  33.896843
